In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils import data
from torch import nn, optim
from tqdm import tqdm
import torch.nn.functional as F
from collections import OrderedDict
import pandas as pd
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [ ]:
# 定义变量路径
CHANGE_DATA_PATH = r"H:\7.Eco_parameter\Hunan_LULC\Sample\change.npy"
STABLE_DATA_PATH = r"H:\7.Eco_parameter\Hunan_LULC\Sample\balanced_stable.npy"
WEIGHT_SAVE_PATH = r"H:\7.Eco_parameter\Hunan_LULC\Best_model\MTUTC.pth"

BATCH_SIZE  = 1024
EPOCH       = 500
NUM_CLASSES = 6      # Water, Forest, Grass, Bare, Impervious Surface, Cropland
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f'使用设备: {DEVICE}')

In [ ]:
# 计算均值和标准差，便于归一化
stable_data = np.load(STABLE_DATA_PATH, allow_pickle=True)
change_data = np.load(CHANGE_DATA_PATH, allow_pickle=True)
data0 = np.concatenate((stable_data, change_data), axis=0)

bands_data = data0[:, :6, :].astype(np.float32)
mean = np.mean(bands_data, axis=(0, 2))
std  = np.std(bands_data,  axis=(0, 2))

print("均值:", mean)
print("标准差:", std)

In [ ]:
# =================== 数据集定义 ===================
# 与原始代码相比，新增返回 cd_label（类别转移矩阵标签）
# 数据格式：Samples * Features(6波段) * Time Series(34年)
# 标签格式：每个时间步的土地覆盖类别（0-5）

class MTUTCDataset(data.Dataset):
    """
    MTUTC 多任务数据集
    返回：
      x        (6, 34)      六波段归一化时间序列
      cls_label (34,)        每个时间步的分类标签（用于 L_cls）
      cd_label  (34, 6, 6)   类别转移矩阵标签（用于 L_cd）
                              cd_label[t, i, j]=1 表示 t 时刻从类别 i 转移到类别 j
                              t=0 时无前驱，全置 0
    """
    def __init__(self, dataset, model_type):
        super(MTUTCDataset, self).__init__()
        self.dataset    = dataset
        self.model_type = model_type   # 保留字段，便于后续扩展数据增强

    def __getitem__(self, index):
        x, label = self.dataset[index, :-1], self.dataset[index, -1]

        # 标准化
        x = ((x - mean[:, None]) / std[:, None]).astype(np.float32)
        label = label.astype(np.int64)   # (34,)

        # 构造类别转移矩阵标签 (34, 6, 6)
        T = label.shape[0]
        cd_label = np.zeros((T, NUM_CLASSES, NUM_CLASSES), dtype=np.float32)
        for t in range(1, T):
            from_cls = label[t - 1]
            to_cls   = label[t]
            cd_label[t, from_cls, to_cls] = 1.0

        return (torch.tensor(x,        dtype=torch.float32),
                torch.tensor(label,    dtype=torch.long),
                torch.tensor(cd_label, dtype=torch.float32))

    def __len__(self):
        return self.dataset.shape[0]


def load_data(batch_size):
    # 加载数据
    change_sample = np.load(CHANGE_DATA_PATH, allow_pickle=True)[:, :7]
    stable_sample = np.load(STABLE_DATA_PATH, allow_pickle=True)[:, :7]
    np.random.shuffle(change_sample)
    np.random.shuffle(stable_sample)

    change_train, change_test = change_sample[:-2000], change_sample[-2000:]
    stable_train, stable_test = stable_sample[:-5000], stable_sample[-5000:]

    print(change_train.shape, stable_train.shape)
    train_arr = np.concatenate((change_train, stable_train), axis=0)
    test_arr  = np.concatenate((change_test,  stable_test),  axis=0)

    train_ds = MTUTCDataset(dataset=train_arr, model_type='train')
    test_ds  = MTUTCDataset(dataset=test_arr,  model_type='test')

    train_dl = data.DataLoader(dataset=train_ds, batch_size=batch_size, shuffle=True)
    test_dl  = data.DataLoader(dataset=test_ds,  batch_size=batch_size)

    return train_dl, test_dl


train_dl, test_dl = load_data(BATCH_SIZE)

In [ ]:
# =================== MTUTC 模型定义 ===================
# 架构：
#   共享 UNet骨干 → 共享特征 [B, 64, 34]
#     ├─ 分类头：1×1 Conv → [B, 6, 34]          （L_cls: 交叉熵）
#     └─ CD头：Concat(Features[64], Classes[6])
#              → Conv → 转移预测 → [B, 34, 6, 6]  （L_cd: 二值交叉熵）
#   L_total = L_cls + L_cd

class MTUTC(nn.Module):
    """Multi-task UNet for Time-series Classification and Change Detection"""

    def __init__(self, in_channels=6, num_classes=6):
        super(MTUTC, self).__init__()
        n = 6
        self.num_classes = num_classes

        # ── UNet 骨干 ──────────────────────────────
        self.encoder1   = self.conv_block(in_channels, 2**n)
        self.encoder2   = self.conv_block(2**n,        2**(n+1))
        self.encoder3   = self.conv_block(2**(n+1),    2**(n+2))
        self.encoder4   = self.conv_block(2**(n+2),    2**(n+3))
        self.bottleneck = self.conv_block(2**(n+3),    2**(n+4))

        self.upconv4  = nn.ConvTranspose1d(2**(n+4), 2**(n+3), kernel_size=2, stride=2)
        self.decoder4 = self.conv_block(2**(n+4), 2**(n+3))
        self.upconv3  = nn.ConvTranspose1d(2**(n+3), 2**(n+2), kernel_size=2, stride=2)
        self.decoder3 = self.conv_block(2**(n+3), 2**(n+2))
        self.upconv2  = nn.ConvTranspose1d(2**(n+2), 2**(n+1), kernel_size=2, stride=2)
        self.decoder2 = self.conv_block(2**(n+2), 2**(n+1))
        self.upconv1  = nn.ConvTranspose1d(2**(n+1), 2**n,     kernel_size=2, stride=2)
        self.decoder1 = self.conv_block(2**(n+1), 2**n)
        # 骨干输出维度: feat (B, 64, L)

        # ── 任务头一：分类头（Classification Head）───────────────
        # 1×1 Conv → 输出每个时间步的 6 类预测
        self.cls_head = nn.Conv1d(2**n, num_classes, kernel_size=1)

        # ── 任务头二：变化检测头（Change Detection Head）──────────
        # 输入 = Concat(骨干特征[64], 分类Softmax概率[6]) → 通道数 = 70
        # 输出 = 类别转移矩阵 (B, 36, L) → reshape → (B, L, 6, 6)
        cd_in = 2**n + num_classes   # 64 + 6 = 70
        self.cd_head = nn.Sequential(
            nn.Conv1d(cd_in, 2**n, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(2**n, num_classes * num_classes, kernel_size=1)
        )

    def conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        # ── UNet 骨干 ──
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(F.max_pool1d(enc1, 2))
        enc3 = self.encoder3(F.max_pool1d(enc2, 2))
        enc4 = self.encoder4(F.max_pool1d(enc3, 2))
        btn  = self.bottleneck(F.max_pool1d(enc4, 2))

        d4 = self.upconv4(btn)
        d4 = F.interpolate(d4, size=enc4.size()[2:], mode='nearest')
        d4 = self.decoder4(torch.cat((d4, enc4), dim=1))

        d3 = self.upconv3(d4)
        d3 = F.interpolate(d3, size=enc3.size()[2:], mode='nearest')
        d3 = self.decoder3(torch.cat((d3, enc3), dim=1))

        d2 = self.upconv2(d3)
        d2 = F.interpolate(d2, size=enc2.size()[2:], mode='nearest')
        d2 = self.decoder2(torch.cat((d2, enc2), dim=1))

        d1 = self.upconv1(d2)
        d1 = F.interpolate(d1, size=enc1.size()[2:], mode='nearest')
        feat = self.decoder1(torch.cat((d1, enc1), dim=1))   # (B, 64, L)

        # ── 分类头 ──
        cls_out  = self.cls_head(feat)                        # (B, 6, L)

        # ── 变化检测头：Concat(特征, 分类Softmax概率) ──
        # 用 softmax 概率而非 logits，传递更平滑的分类信号给 CD 头
        cls_prob = F.softmax(cls_out, dim=1)                  # (B, 6, L)
        cd_in    = torch.cat([feat, cls_prob], dim=1)         # (B, 70, L)
        cd_raw   = self.cd_head(cd_in)                        # (B, 36, L)

        B, _, L  = cd_raw.shape
        cd_out   = cd_raw.permute(0, 2, 1).reshape(
                       B, L, self.num_classes, self.num_classes)  # (B, 34, 6, 6)

        return cls_out, cd_out


model = MTUTC(in_channels=6, num_classes=NUM_CLASSES).to(DEVICE)
print('MTUTC 模型参数量：', sum(p.numel() for p in model.parameters()))

# 输出形状验证
_x = torch.randn(4, 6, 34).to(DEVICE)
_cls, _cd = model(_x)
print(f'分类头输出: {_cls.shape}')    # 期望 (4, 6, 34)
print(f'CD头输出:   {_cd.shape}')     # 期望 (4, 34, 6, 6)
del _x, _cls, _cd

In [ ]:
# =================== 评价指标定义 ===================

class ClassificationEvaluator(object):
    """土地覆盖分类指标"""
    def __init__(self, num_class):
        self.num_class = num_class
        self.confusion_matrix = np.zeros((self.num_class,) * 2)

    def Pixel_Accuracy(self):   # OA
        Acc = np.diag(self.confusion_matrix).sum() / self.confusion_matrix.sum()
        return Acc

    def Pixel_Accuracy_Class(self):   # AA
        Acc_classes = np.diag(self.confusion_matrix) / self.confusion_matrix.sum(axis=1)
        Acc = np.nanmean(Acc_classes)
        return Acc_classes, Acc

    def Kappa(self):
        p_o = self.Pixel_Accuracy()
        pre   = np.sum(self.confusion_matrix, axis=0)
        label = np.sum(self.confusion_matrix, axis=1)
        p_e = (pre * label).sum() / (self.confusion_matrix.sum() ** 2)
        kappa = (p_o - p_e) / (1 - p_e)
        return kappa

    def Overall_Accuracy(self):
        return self.Pixel_Accuracy()

    def _generate_matrix(self, gt_image, pre_image):
        mask  = (gt_image >= 0) & (gt_image < self.num_class)
        label = self.num_class * gt_image[mask].astype('int') + pre_image[mask]
        count = np.bincount(label, minlength=self.num_class ** 2)
        return count.reshape(self.num_class, self.num_class)

    def add_batch(self, gt_image, pre_image):
        assert gt_image.shape == pre_image.shape
        self.confusion_matrix += self._generate_matrix(gt_image, pre_image)

    def reset(self):
        self.confusion_matrix = np.zeros((self.num_class,) * 2)


eps = np.finfo(np.float32).eps.item()

class SpatialChangeDetectScore(object):
    """空间域变化检测精度"""
    def __init__(self):
        self.spatial_f1 = None
        self.spatial_ua_Nochange = None
        self.spatial_ua_change   = None
        self.spatial_pa_Nochange = None
        self.spatial_pa_change   = None
        self.PreChange_LabChange     = eps
        self.PreNoChange_LabChange   = eps
        self.PreChange_LabNoChange   = eps
        self.PreNoChange_LabNoChange = eps

    def addValue(self, label, pre):
        if   len(label) != 0 and len(pre) != 0: self.PreChange_LabChange     += 1
        elif len(label) == 0 and len(pre) == 0: self.PreNoChange_LabNoChange += 1
        elif len(label) == 0 and len(pre) != 0: self.PreChange_LabNoChange   += 1
        elif len(label) != 0 and len(pre) == 0: self.PreNoChange_LabChange   += 1

    def getScore(self):
        self.spatial_pa_change   = self.PreChange_LabChange / (self.PreChange_LabChange + self.PreChange_LabNoChange)
        self.spatial_pa_Nochange = self.PreNoChange_LabNoChange / (self.PreNoChange_LabNoChange + self.PreNoChange_LabChange)
        self.spatial_ua_change   = self.PreChange_LabChange / (self.PreChange_LabChange + self.PreNoChange_LabChange)
        self.spatial_ua_Nochange = self.PreNoChange_LabNoChange / (self.PreNoChange_LabNoChange + self.PreChange_LabNoChange)
        self.spatial_pa = self.spatial_pa_change
        self.spatial_ua = self.spatial_ua_change
        self.spatial_f1 = 2 * self.spatial_pa * self.spatial_ua / (self.spatial_pa + self.spatial_ua)


class TemporalChangeDetectScore(object):
    """时间域变化检测精度"""
    def __init__(self, series_length, error_rate=0):
        self.temporal_f1 = None
        self.temporal_ua_Nochange = None
        self.temporal_ua_change   = None
        self.temporal_pa_Nochange = None
        self.temporal_pa_change   = None
        self.PreChange_LabChange     = eps
        self.PreNoChange_LabChange   = eps
        self.PreChange_LabNoChange   = eps
        self.PreNoChange_LabNoChange = eps
        self.series_length = series_length
        self.error_rate    = error_rate

    def addValue(self, label, pre):
        for lab in label:
            for p_index in range(len(pre)):
                if abs(pre[p_index] - lab) <= self.error_rate:
                    pre[p_index] = lab
        better_pre = list(set(pre))
        hot_label = np.zeros(self.series_length)
        hot_pre   = np.zeros(self.series_length)
        if len(label)      != 0: hot_label[np.array(label)]      = 1
        if len(better_pre) != 0: hot_pre[np.array(better_pre)]   = 1
        self.PreChange_LabChange     += np.where((hot_pre == 1) & (hot_label == 1))[0].shape[0]
        self.PreNoChange_LabChange   += np.where((hot_pre != 1) & (hot_label == 1))[0].shape[0]
        self.PreChange_LabNoChange   += np.where((hot_pre == 1) & (hot_label != 1))[0].shape[0]
        self.PreNoChange_LabNoChange += np.where((hot_pre != 1) & (hot_label != 1))[0].shape[0]

    def getScore(self):
        self.temporal_pa_change   = self.PreChange_LabChange / (self.PreChange_LabChange + self.PreChange_LabNoChange)
        self.temporal_pa_Nochange = self.PreNoChange_LabNoChange / (self.PreNoChange_LabNoChange + self.PreNoChange_LabChange)
        self.temporal_ua_change   = self.PreChange_LabChange / (self.PreChange_LabChange + self.PreNoChange_LabChange)
        self.temporal_ua_Nochange = self.PreNoChange_LabNoChange / (self.PreNoChange_LabNoChange + self.PreChange_LabNoChange)
        self.temporal_pa = self.temporal_pa_change
        self.temporal_ua = self.temporal_ua_change
        self.temporal_f1 = 2 * self.temporal_pa * self.temporal_ua / (self.temporal_pa + self.temporal_ua)

In [ ]:
# =================== 损失函数与优化器 ===================
# L_total = L_cls + L_cd

loss_cls = nn.CrossEntropyLoss()      # L_cls：分类交叉熵
loss_cd  = nn.BCEWithLogitsLoss()     # L_cd ：转移矩阵二值交叉熵
optimizer = optim.Adam(params=model.parameters(), lr=0.001)

# 评价指标
evaluator = ClassificationEvaluator(NUM_CLASSES)

# 记录训练过程指标
train_loss_list = []
test_loss_list  = []
test_classification_OA    = []
test_classification_Kappa = []
test_change_detection_spatial_f1  = []
test_change_detection_temporal_f1 = []
test_overall_accuracy = []

best_test_loss                        = 999
best_test_classification_Kappa        = -999
best_test_change_detection_spatial_f1 = -999
best_test_change_detection_temporal_f1= -999
best_test_overall_accuracy            = -999

In [ ]:
# =================== 训练与测试主循环 ===================

for epoch in range(EPOCH):
    evaluator.reset()
    spatialscore  = SpatialChangeDetectScore()
    temporalscore = TemporalChangeDetectScore(34, error_rate=3)   # 容忍误差±3年

    # ── 训练阶段 ─────────────────────────────────────────────────────────
    model.train()
    train_tqdm = tqdm(iterable=train_dl, total=len(train_dl))
    train_tqdm.set_description_str('Train epoch: {:3d}'.format(epoch))
    train_loss_sum = torch.tensor(data=[], dtype=torch.float, device=DEVICE)

    for train_x, train_cls_y, train_cd_y in train_tqdm:
        train_x      = train_x.to(DEVICE)
        train_cls_y  = train_cls_y.to(DEVICE)    # (B, 34)      分类标签
        train_cd_y   = train_cd_y.to(DEVICE)     # (B, 34, 6, 6) 转移矩阵标签

        cls_pred, cd_pred = model(train_x)        # (B, 6, 34) / (B, 34, 6, 6)

        # 联合损失：L_total = L_cls + L_cd
        L_cls   = loss_cls(cls_pred, train_cls_y)
        L_cd    = loss_cd(cd_pred, train_cd_y)
        L_total = L_cls + L_cd

        optimizer.zero_grad()
        L_total.backward()
        optimizer.step()

        with torch.no_grad():
            train_loss_sum = torch.cat(
                [train_loss_sum, torch.unsqueeze(L_total, dim=-1)], dim=-1)
            train_tqdm.set_postfix({'train loss': train_loss_sum.mean().item()})

    train_tqdm.close()
    train_loss_list.append(train_loss_sum.mean().item())

    # ── 测试阶段 ─────────────────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        test_loss_sum = torch.tensor(data=[], dtype=torch.float, device=DEVICE)

        for test_x, test_cls_y, test_cd_y in test_dl:
            test_x     = test_x.to(DEVICE)
            test_cls_y = test_cls_y.to(DEVICE)
            test_cd_y  = test_cd_y.to(DEVICE)

            cls_pred, cd_pred = model(test_x)

            L_cls   = loss_cls(cls_pred, test_cls_y)
            L_cd    = loss_cd(cd_pred, test_cd_y)
            L_total = L_cls + L_cd
            test_loss_sum = torch.cat(
                [test_loss_sum, torch.unsqueeze(L_total, dim=-1)], dim=-1)

            # ── 分类精度评估（来自分类头）──
            cls_preds  = torch.argmax(cls_pred, dim=1).cpu().numpy()   # (B, 34)
            cls_labels = test_cls_y.cpu().numpy()                       # (B, 34)
            evaluator.add_batch(cls_labels, cls_preds)

            # ── 变化检测精度评估（来自 CD 头）──
            # 对每个时间步的 6×6 转移矩阵取 sigmoid 后找最大概率的转移
            # 若 argmax 对应的 from_class ≠ to_class 则判定为变化
            cd_prob = torch.sigmoid(cd_pred).cpu().numpy()   # (B, 34, 6, 6)
            for b in range(cd_prob.shape[0]):
                pre_change_date   = []
                label_change_date = []
                for t in range(1, 34):
                    # 预测变化点：CD头判断
                    flat_idx  = np.argmax(cd_prob[b, t])   # 0~35
                    from_pred = flat_idx // NUM_CLASSES
                    to_pred   = flat_idx %  NUM_CLASSES
                    if from_pred != to_pred:
                        pre_change_date.append(t)
                    # 真实变化点：从分类标签判断
                    if cls_labels[b, t] != cls_labels[b, t - 1]:
                        label_change_date.append(t)

                spatialscore.addValue(pre_change_date, label_change_date)
                temporalscore.addValue(pre_change_date, label_change_date)

    # ── 汇总本轮指标 ──────────────────────────────────────────────────────
    spatialscore.getScore()
    temporalscore.getScore()

    test_loss_list.append(test_loss_sum.mean().item())
    test_classification_OA.append(evaluator.Pixel_Accuracy())
    test_classification_Kappa.append(evaluator.Kappa())
    test_change_detection_spatial_f1.append(spatialscore.spatial_f1)
    test_change_detection_temporal_f1.append(temporalscore.temporal_f1)
    test_overall_accuracy.append(evaluator.Overall_Accuracy())

    # ── 保存最优模型（以时间域变化检测 F1 最高为准）──────────────────────
    if temporalscore.temporal_f1 > best_test_change_detection_temporal_f1:
        torch.save(model.state_dict(), WEIGHT_SAVE_PATH)
        best_test_loss                         = test_loss_sum.mean().item()
        best_test_classification_Kappa         = round(evaluator.Kappa(), 4)
        best_test_change_detection_spatial_f1  = round(spatialscore.spatial_f1, 4)
        best_test_change_detection_temporal_f1 = round(temporalscore.temporal_f1, 4)
        best_test_overall_accuracy             = round(evaluator.Overall_Accuracy(), 4)
        best_spatialscore  = spatialscore
        best_temporalscore = temporalscore
        best_evaluator = ClassificationEvaluator(NUM_CLASSES)
        best_evaluator.confusion_matrix = evaluator.confusion_matrix.copy()

# =================== 输出最终精度 ===================
_, aa = best_evaluator.Pixel_Accuracy_Class()
print('\n最终模型精度：')
print(f'Overall Accuracy (OA) : {best_test_overall_accuracy}')
print(f'Average Accuracy (AA) : {round(aa, 4)}')
print(f'Kappa                 : {best_test_classification_Kappa}')
print(f'Spatial PA            : {round(best_spatialscore.spatial_pa_change, 4)}')
print(f'Spatial UA            : {round(best_spatialscore.spatial_ua_change, 4)}')
print(f'Spatial F1            : {best_test_change_detection_spatial_f1}')
print(f'Temporal PA           : {round(best_temporalscore.temporal_pa_change, 4)}')
print(f'Temporal UA           : {round(best_temporalscore.temporal_ua_change, 4)}')
print(f'Temporal F1           : {best_test_change_detection_temporal_f1}')
print('\n最终混淆矩阵：')
print(pd.DataFrame(best_evaluator.confusion_matrix))

In [ ]:
# =================== 训练过程可视化 ===================

plt.figure(figsize=(12, 5))

# 损失曲线
plt.subplot(1, 2, 1)
plt.plot(train_loss_list, label='Train Loss', color='blue')
plt.plot(test_loss_list,  label='Test Loss',  color='orange')
plt.title('Loss (L_cls + L_cd)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid()

# 精度曲线
plt.subplot(1, 2, 2)
plt.plot(test_classification_OA,           label='OA',          color='green')
plt.plot(test_classification_Kappa,        label='Kappa',        color='red')
plt.plot(test_change_detection_spatial_f1, label='Spatial F1',  color='purple')
plt.plot(test_change_detection_temporal_f1,label='Temporal F1', color='brown')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()